# Complete case analysis
Rebuilt for pandas 3 and Plotly 7. Data: `data/data_science_job.csv.gz`, a gzip-compressed copy of the job-applicant file (19,158 rows, 13 columns) from Kaggle's *HR Analytics: Job Change of Data Scientists*. pandas reads `.gz` files directly.

In [ ]:
# pandas for the table, numpy for numbers, Plotly for charts, scipy for the KDE curves
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

## 1. How much of each column is missing

In [ ]:
# load the data and look at it
df = pd.read_csv("data/data_science_job.csv.gz")
print(df.shape)
df.head()

In [ ]:
# isnull() marks every missing cell True; mean() of True/False is the share of True
(df.isnull().mean() * 100).round(2)

## 2. Choose the columns for complete case analysis

In [ ]:
# keep the columns that have some missing values, but fewer than 5%
cols = [c for c in df.columns if 0 < df[c].isnull().mean() < 0.05]
cols

In [ ]:
# share of rows left if we drop every row with a gap in one of these columns
len(df.dropna(subset=cols)) / len(df)

In [ ]:
# for comparison: dropping rows with a gap in ANY column keeps less than half
len(df.dropna()) / len(df)

In [ ]:
# complete case analysis: drop the incomplete rows, keep all 13 columns
cca = df.dropna(subset=cols)
df.shape, cca.shape

## 3. Numerical columns: compare the distributions before and after

In [ ]:
# overlay the histogram before (red) and after (green); density makes the two heights comparable
def compare_hist(col):
    fig = go.Figure()
    for data, colour, name in [(df[col], "#E45756", "before CCA"), (cca[col], "#54A24B", "after CCA")]:
        fig.add_trace(go.Histogram(x=data, nbinsx=50, histnorm="probability density",
                                   marker_color=colour, opacity=0.55, name=name))
    fig.update_layout(barmode="overlay", title=col, template="simple_white")
    fig.show()

for col in ["training_hours", "city_development_index", "experience"]:
    compare_hist(col)

In [ ]:
# the same comparison as smooth KDE curves; if the two curves lie on top of each other, CCA kept the shape
def compare_kde(col):
    grid = np.linspace(df[col].min(), df[col].max(), 300)
    fig = go.Figure()
    for data, colour, name in [(df[col], "#E45756", "before CCA"), (cca[col], "#54A24B", "after CCA")]:
        fig.add_trace(go.Scatter(x=grid, y=gaussian_kde(data.dropna())(grid), mode="lines",
                                 line=dict(color=colour, width=3), name=name))
    fig.update_layout(title=col, template="simple_white")
    fig.show()

for col in ["training_hours", "city_development_index", "experience"]:
    compare_kde(col)

In [ ]:
# the same check in numbers: mean and median before and after
num = ["training_hours", "city_development_index", "experience"]
before = df[num].agg(["mean", "median"]).T.add_suffix(" before")
after = cca[num].agg(["mean", "median"]).T.add_suffix(" after")
pd.concat([before, after], axis=1).round(3)

## 4. Categorical columns: compare the share of each category

In [ ]:
# value_counts(normalize=True) gives each category's share of the non-missing values
def compare_shares(col):
    temp = pd.concat([df[col].value_counts(normalize=True),
                      cca[col].value_counts(normalize=True)], axis=1)
    temp.columns = ["original", "cca"]
    return temp.round(4)

compare_shares("enrolled_university")

In [ ]:
compare_shares("education_level")

In [ ]:
# careful: dividing by len(df) counts the missing rows too, so the original shares look smaller
(df["enrolled_university"].value_counts() / len(df)).round(4)